# B6: the MLP / SwiGLU

**Goal:** build `down_proj(silu(gate_proj(x)) * up_proj(x))` and match HF.

Attention mixes information between tokens. The MLP transforms each token independently: two branches expand 2048 features to 5632, their elementwise product is projected back to 2048.

**Spec:** input `(..., hidden)` -> same output shape, dtype and device. Three bias-free projections named `gate_proj`, `up_proj`, `down_proj`, compatible with HF weights. `silu(z) = z * sigmoid(z)`.

Run top to bottom in **Colab on a T4**. Tests run first; no runtime success is assumed before your run.

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/guptasameer112/tinyserve.git"
REPO_DIR = "/content/tinyserve"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], check=True, capture_output=True, text=True).stdout)


In [ ]:
import importlib, math
import torch
import tinyserve.models.mlp, tinyserve.probe
for module in (tinyserve.models.mlp, tinyserve.probe):
    importlib.reload(module)
from tinyserve.models.mlp import MLP
from tinyserve.envinfo import collect_env, save_result

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, then rerun.")
DEVICE = "cuda"
torch.backends.cuda.matmul.allow_tf32 = False
torch.set_float32_matmul_precision("highest")
print(collect_env())


## 1. Test first

Hand math, float64 formula, HF weight loading, feature shapes, independent token rows, zero branches and low-precision checks. B4/B5 tests run too because the shared probe now imports the MLP.

A failure stops the notebook. Read skips: CUDA and HF checks should be available in this environment.

In [ ]:
test_run = subprocess.run([sys.executable, "-m", "pytest", "tests/test_b4_attention.py", "tests/test_b5_gqa.py", "tests/test_b6_mlp.py", "-v", "-ra"],
                          capture_output=True, text=True)
print(test_run.stdout)
print(test_run.stderr)
test_run.check_returncode()


## 2. By hand: two features

Input `[1, -1]`; gate projection is identity, up projection doubles the input, down projection is identity.

```
gate          [1, -1]
silu(gate)    [0.731059, -0.268941]
up            [2, -2]
product       [1.462117, 0.537883]
output        [1.462117, 0.537883]
```

The second output is positive because negative times negative is positive. The gate is **not** an attention probability: it can be negative or larger than 1.

In [ ]:
hand_mlp = MLP(2, 2, device=DEVICE, dtype=torch.float32)
with torch.no_grad():
    hand_mlp.gate_proj.weight.copy_(torch.eye(2, device=DEVICE))
    hand_mlp.up_proj.weight.copy_(2 * torch.eye(2, device=DEVICE))
    hand_mlp.down_proj.weight.copy_(torch.eye(2, device=DEVICE))
    hand_input = torch.tensor([[1.0, -1.0]], device=DEVICE)
    hand_output = hand_mlp(hand_input)
print("output:", hand_output.tolist())
expected_output = torch.tensor([[2 / (1 + math.exp(-1)), 2 / (1 + math.exp(1))]], device=DEVICE)
torch.testing.assert_close(hand_output, expected_output, rtol=0, atol=1e-6)


## 3. Shape tour and independent tokens

Use a small synthetic MLP to keep the example cheap. Unlike attention, a row does not read any other row. Changing the last token must leave the earlier outputs unchanged.

For TinyLlama, the three matrices contain `3 * 2048 * 5632 = 34,603,008` weights per layer. Across 22 layers: 761,266,176 weights, about **69.2%** of the model. Weight share is not automatically execution-time share.

In [ ]:
torch.manual_seed(6)
demo_mlp = MLP(64, 128, device=DEVICE, dtype=torch.float32).eval()
demo_input = torch.randn(4, 64, device=DEVICE)
with torch.no_grad():
    demo_gate = demo_mlp.gate_proj(demo_input)
    demo_up = demo_mlp.up_proj(demo_input)
    demo_output = demo_mlp(demo_input)
    changed_input = demo_input.clone()
    changed_input[-1] += 10
    changed_output = demo_mlp(changed_input)
row_independence_diff = (demo_output[:-1] - changed_output[:-1]).abs().max().item()
print("input:", tuple(demo_input.shape), "gate/up:", tuple(demo_gate.shape), "output:", tuple(demo_output.shape))
print("earlier tokens changed by:", row_independence_diff)
assert row_independence_diff == 0.0
mlp_params_per_layer = 3 * 2048 * 5632
print(f"TinyLlama MLP params per layer: {mlp_params_per_layer:,}; all 22: {22 * mlp_params_per_layer:,}")


## 4. FP16 vs FP32 on the same rounded weights and inputs

Three projection outputs, SiLU and the elementwise product involve intermediate rounding. Unlike B4's upcast reference, the MLP follows HF's native operation order.

For these controlled small inputs use `rtol=atol=5e-3`. Cancellation can make the final output close to zero, so a relative-only check is insufficient. This is a test bound, not a universal guarantee for arbitrary activations. Real-model parity below uses FP32 and a tighter tolerance.

In [ ]:
half_mlp = MLP(64, 128, device=DEVICE, dtype=torch.float16).eval()
half_mlp.load_state_dict(demo_mlp.state_dict(), strict=True)
rounded_reference = MLP(64, 128, device=DEVICE, dtype=torch.float32).eval()
rounded_reference.load_state_dict({name: weight.float() for name, weight in half_mlp.state_dict().items()}, strict=True)
half_input = demo_input.half()
with torch.no_grad():
    half_output = half_mlp(half_input)
    float_output = rounded_reference(half_input.float())
fp16_max_diff = (half_output.float() - float_output).abs().max().item()
print("FP16 vs FP32 max difference:", fp16_max_diff)
torch.testing.assert_close(half_output.float(), float_output, rtol=5e-3, atol=5e-3)


## 5. All 22 real TinyLlama MLPs

Load the model in FP32, capture each HF MLP's input and output, copy its trained weights into our module, and compare. The shared hook helper removes hooks even if the forward pass fails.

The probe asserts `rtol=atol=1e-5` for every MLP. This validates the MLP in isolation; residual additions and a full decoder come in B7.

FP32 weights use about 4.4 GB. Close other GPU-heavy notebook kernels if memory is low.

In [ ]:
from tinyserve.basics import load_model, encode
from tinyserve.probe import compare_mlp
tokenizer, model = load_model(device=DEVICE, dtype=torch.float32)
model.set_attn_implementation("eager")
prompt_ids = encode(tokenizer, "The capital of France is", DEVICE)
mlp_rows = compare_mlp(model, prompt_ids)
assert len(mlp_rows) == model.config.num_hidden_layers == 22
for row in mlp_rows:
    print(f"{row['name']}: max |diff| {row['max_abs_diff']:.2e}, "
          f"input max {row['input_max_abs']:.3f}, output max {row['output_max_abs']:.3f}")
print("gate weight shape:", tuple(model.model.layers[0].mlp.gate_proj.weight.shape))


## 6. Save evidence and three interview questions

1. **What is SwiGLU?** `down(silu(gate(x)) * up(x))`.
2. **Does it mix tokens?** No: only features within each token.
3. **Why so many weights?** Three large projection matrices per layer, with intermediate width 5632 versus hidden width 2048.

The min/max below is the range across tested layers, not repeated timing trials. Save the notebook with its outputs and retain the printed JSON after a successful run.

**Next: B7 decoder layer and residuals.** Connect RMSNorm, RoPE, attention/GQA and the MLP into one layer.

In [ ]:
layer_differences = [row["max_abs_diff"] for row in mlp_rows]
saved_path = save_result("B6_mlp", {
    "hidden_size": 2048, "intermediate_size": 5632,
    "mlp_params_per_layer": mlp_params_per_layer,
    "mlp_params_all_layers": 22 * mlp_params_per_layer,
    "hand_output": hand_output.tolist(),
    "row_independence_max_abs_diff": row_independence_diff,
    "fp16_vs_fp32_max_abs_diff": fp16_max_diff,
    "real_model_dtype": "float32",
    "real_model_comparison": mlp_rows,
    "real_model_diff_range_across_layers": {"min": min(layer_differences), "max": max(layer_differences)},
    "pytest": {"returncode": test_run.returncode, "stdout": test_run.stdout, "stderr": test_run.stderr},
})
print(saved_path)
print(saved_path.read_text())


In [ ]:
import gc
del model
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory still held by this kernel: {torch.cuda.memory_allocated() / 2**20:.0f} MiB")
